In [1]:
# CELL 01 — Set stock ticker
NAME = 'ORI'

In [2]:
# CELL 02 — Imports + DB engines
import json
import os
import pandas as pd
from datetime import date, datetime, timedelta
from sqlalchemy import create_engine, text

engine = create_engine('mysql+pymysql://root:@localhost:3306/portfolio_development')
conpf = engine.connect()

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

format_dict = {'amt':'{:,.2f}','net':'{:,.2f}','refund':'{:,.2f}',
              'grs_amt':'{:,.2f}','net_amt':'{:,.2f}','number':'{:,}',
               'ppu':'{:,.4f}'}

today = date.today()
dd = today.day 

In [3]:
# CELL 03 — Resolve paths
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")
#C:\Users\PC1\OneDrive\A5\Excel
csv_path = os.path.join(base_path, "CSV")

In [4]:
# CELL 04 — Print path check
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Data path (dat_path): {dat_path}") 
print(f"CSV path (csv_path): {csv_path}") 
print(f"Obsidian path (osd_path): {osd_path}")

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Daily
Base path: C:\Users\PC1\OneDrive\A4
Data path (dat_path): C:\Users\PC1\OneDrive\A4\Data
CSV path (csv_path): C:\Users\PC1\OneDrive\A4\CSV
Obsidian path (osd_path): C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


In [5]:
# CELL 05 — Load portfolio + price data
# =====================================================================
# PORTFOLIO DASHBOARD DATA COLLECTION
# Reuses NAME, conpf, const, csv_path from earlier cells
# =====================================================================

# --- Buys (already fetched as df_buys in your notebook, refetch for safety) ---
sql_buys = f"""
SELECT B.id, B.stock_id, B.date, B.qty, B.price, B.fee, B.vat,
       B.status, B.net, B.kind, S.name
FROM buys B
JOIN stocks S ON B.stock_id = S.id
WHERE S.name = '{NAME}'
ORDER BY B.date
"""
df_buys_dash = pd.read_sql(sql_buys, con=conpf)

# --- Sells joined through buys -> stocks ---
sql_sells = f"""
SELECT E.id, E.buy_id, E.date AS sell_date, E.price AS sell_price,
       E.net AS proceeds, E.days, E.profit, E.percent,
       B.date AS buy_date, B.qty, B.price AS buy_price, B.net AS cost, S.name
FROM sells E
JOIN buys B ON E.buy_id = B.id
JOIN stocks S ON B.stock_id = S.id
WHERE S.name = '{NAME}'
ORDER BY E.date
"""
df_sells_dash = pd.read_sql(sql_sells, con=conpf)

# --- Dividends ---
sql_div = f"""
SELECT D.year, D.quarter, D.number, D.ppu, D.amt, D.net, D.x_date, D.p_date, S.name
FROM dividends D
JOIN stocks S ON D.stock_id = S.id
WHERE S.name = '{NAME}'
ORDER BY D.p_date
"""
df_div_dash = pd.read_sql(sql_div, con=conpf)

# --- Sanity check: net should equal gross × (1 − withholding) ---
if not df_div_dash.empty:
    df_div_dash["expected_net"] = (df_div_dash["amt"] * 0.90).round(2)
    df_div_dash["delta"] = (df_div_dash["net"] - df_div_dash["expected_net"]).abs()
    suspicious = df_div_dash[df_div_dash["delta"] > 1.00]

    if not suspicious.empty:
        print("⚠️  Dividends where net ≠ 90% of gross:")
        print(suspicious[["year", "quarter", "amt", "net", "expected_net", "delta"]])
    else:
        print("✅ All dividend net amounts match 90% of gross")

# --- 2 years of daily prices ---
sql_rcl = f"""
SELECT date, price, maxp, minp, qty, opnp
FROM price
WHERE name = '{NAME}'
  AND date >= CURDATE() - INTERVAL 3 YEAR
ORDER BY date
"""
df_rcl_dash = pd.read_sql(sql_rcl, con=const)
df_rcl_dash["date"] = pd.to_datetime(df_rcl_dash["date"]).dt.strftime("%Y-%m-%d")

# --- SET Index ---
sql_set = """
SELECT date, setindex
FROM setindex
WHERE date >= CURDATE() - INTERVAL 3 YEAR
ORDER BY date
"""
df_set_dash = pd.read_sql(sql_set, con=const)
df_set_dash["date"] = pd.to_datetime(df_set_dash["date"]).dt.strftime("%Y-%m-%d")

print(f"Buys:      {df_buys_dash.shape}")
print(f"Sells:     {df_sells_dash.shape}")
print(f"Dividends: {df_div_dash.shape}")
print(f"RCL price: {df_rcl_dash.shape}")
print(f"SET index: {df_set_dash.shape}")

# Latest price for unrealized P&L
LATEST_PRICE = float(df_rcl_dash.iloc[-1]["price"])
LATEST_DATE  = df_rcl_dash.iloc[-1]["date"]
print(f"\nLatest {NAME} close: ฿{LATEST_PRICE:.2f} on {LATEST_DATE}")

✅ All dividend net amounts match 90% of gross
Buys:      (30, 11)
Sells:     (26, 13)
Dividends: (17, 11)
RCL price: (722, 6)
SET index: (720, 2)

Latest ORI close: ฿1.78 on 2026-10-27


In [6]:
# CELL 06 — Compute metrics + build payload
# =====================================================================
# PORTFOLIO METRICS + JSON SERIALIZATION
# Adaptive MA window — works for tickers with 0, 75, or 3,000 price rows
# =====================================================================

import json

# --- Active buys ---
active = df_buys_dash[df_buys_dash["status"] == "Active"].copy()
active_cost = float(active["net"].sum()) if not active.empty else 0.0
active_qty  = int(active["qty"].sum())   if not active.empty else 0
market_value = active_qty * LATEST_PRICE
unrealized   = market_value - active_cost
unrealized_pct = (unrealized / active_cost * 100) if active_cost else 0

# --- Realized ---
realized_profit = float(df_sells_dash["profit"].sum()) if not df_sells_dash.empty else 0.0

# --- Dividends ---
total_dividends = float(df_div_dash["net"].sum()) if not df_div_dash.empty else 0.0

# --- Total return ---
total_return = realized_profit + total_dividends + unrealized

# =====================================================================
# ADAPTIVE MA WINDOW
# =====================================================================
MA_WINDOW_MAX = 200     # ideal
MA_WINDOW_MIN = 20      # absolute minimum for a usable trend read

prices = df_rcl_dash["price"].astype(float).tolist()
n_rows = len(prices)

if n_rows >= MA_WINDOW_MAX:
    MA_WINDOW = MA_WINDOW_MAX
    ma_label  = f"{MA_WINDOW}-day MA"
elif n_rows >= MA_WINDOW_MIN:
    MA_WINDOW = n_rows - 5   # leave 5 rows so the first MA value is stable
    ma_label  = f"{MA_WINDOW}-day MA (short — only {n_rows} rows available)"
    print(f"⚠️  Only {n_rows} rows of price data — using {MA_WINDOW}-day MA instead of 200")
else:
    MA_WINDOW = None
    ma_label  = f"N/A — only {n_rows} rows (need ≥ {MA_WINDOW_MIN})"
    print(f"⚠️  Only {n_rows} rows of price data — too few for any useful MA")

# --- Build MA series ---
ma_series = []
if MA_WINDOW:
    for i in range(len(prices)):
        if i < MA_WINDOW - 1:
            ma_series.append(None)
        else:
            ma_series.append(round(sum(prices[i - MA_WINDOW + 1 : i + 1]) / MA_WINDOW, 4))
else:
    ma_series = [None] * len(prices)

last_ma = next((v for v in reversed(ma_series) if v is not None), None)

# =====================================================================
# MA DIRECTION + DISTANCE (guarded)
# =====================================================================
if last_ma:
    ma_distance = ((LATEST_PRICE - last_ma) / last_ma * 100) if last_ma else 0

    def ma_at(offset):
        idx = len(ma_series) - 1 - offset
        return ma_series[idx] if 0 <= idx < len(ma_series) else None

    ma_now   = last_ma
    ma_20ago = ma_at(20)
    ma_60ago = ma_at(60)

    ma_trend_20d_pct = ((ma_now - ma_20ago) / ma_20ago * 100) if ma_20ago else 0
    ma_trend_60d_pct = ((ma_now - ma_60ago) / ma_60ago * 100) if ma_60ago else 0

    if ma_20ago is None:
        ma_trend, ma_trend_color = "N/A", "#8b9aa8"
    elif ma_trend_20d_pct > 0.5:
        ma_trend, ma_trend_color = "RISING", "#4caf50"
    elif ma_trend_20d_pct < -0.5:
        ma_trend, ma_trend_color = "FALLING", "#ef5350"
    else:
        ma_trend, ma_trend_color = "FLAT", "#ffb74d"

    print(f"\n--- {ma_label} direction ---")
    print(f"  MA now       : ฿{ma_now:.2f}")
    if ma_20ago:
        print(f"  MA 20d ago   : ฿{ma_20ago:.2f}  ({ma_trend_20d_pct:+.2f}%)")
    else:
        print(f"  MA 20d ago   : insufficient data")
    if ma_60ago:
        print(f"  MA 60d ago   : ฿{ma_60ago:.2f}  ({ma_trend_60d_pct:+.2f}%)")
    else:
        print(f"  MA 60d ago   : insufficient data")
    print(f"  Trend        : {ma_trend}")
else:
    ma_distance = 0
    ma_now = ma_20ago = ma_60ago = None
    ma_trend_20d_pct = 0
    ma_trend_60d_pct = 0
    ma_trend, ma_trend_color = "N/A", "#8b9aa8"
    print(f"\n--- MA direction ---")
    print(f"  {ma_label}")

# =====================================================================
# TECHNICAL BUY ZONES (guarded)
# =====================================================================
if last_ma:
    def zone(lo_mult, hi_mult):
        return (round(last_ma * lo_mult, 2), round(last_ma * hi_mult, 2))

    buy_zones = [
        {
            "label":  "Trend-following",
            "sub":    f"Buy on pullback to just above the {MA_WINDOW}-MA",
            "lo":     zone(1.00, 1.05)[0],
            "hi":     zone(1.00, 1.05)[1],
            "color":  "#4caf50",
            "note":   "Healthy uptrend entry. Lowest risk, lowest reward.",
        },
        {
            "label":  "Mean-reversion",
            "sub":    f"Buy when price dips below the {MA_WINDOW}-MA and turns back up",
            "lo":     zone(0.90, 1.00)[0],
            "hi":     zone(0.90, 1.00)[1],
            "color":  "#4fc3f7",
            "note":   "Classic dip-buy zone in an intact long-term trend.",
        },
        {
            "label":  "Deep value",
            "sub":    "Aggressive buy only if the thesis is intact",
            "lo":     zone(0.75, 0.90)[0],
            "hi":     zone(0.75, 0.90)[1],
            "color":  "#ffb74d",
            "note":   "Rare. High reward, but verify fundamentals first.",
        },
    ]

    def classify_price(price, zones):
        for z in zones:
            if z["lo"] <= price <= z["hi"]:
                return f'INSIDE — {z["label"]} zone'
        if price < zones[-1]["lo"]:
            return "BELOW ALL ZONES — capitulation"
        if price > zones[0]["hi"]:
            return "ABOVE ALL ZONES — no buy signal"
        return "Between zones"

    buy_zone_status = classify_price(LATEST_PRICE, buy_zones)

    print(f"\n--- Technical buy zones ({ma_label}) ---")
    for z in buy_zones:
        print(f'  {z["label"]:18s}  ฿{z["lo"]:>8.2f} – ฿{z["hi"]:>8.2f}   {z["sub"]}')
    print(f"  Today (฿{LATEST_PRICE:.2f}): {buy_zone_status}")
else:
    buy_zones = []
    buy_zone_status = "N/A — insufficient price history"
    print(f"\n--- Technical buy zones ---")
    print(f"  {ma_label}")

# =====================================================================
# SUMMARY
# =====================================================================
print(f"\n--- Portfolio summary ---")
print(f"Active cost:    ฿{active_cost:,.2f}")
print(f"Active qty:     {active_qty:,}")
print(f"Market value:   ฿{market_value:,.2f}")
print(f"Unrealized P&L: ฿{unrealized:+,.2f} ({unrealized_pct:+.2f}%)")
print(f"Realized:       ฿{realized_profit:+,.2f}")
print(f"Dividends:      ฿{total_dividends:,.2f}")
print(f"Total return:   ฿{total_return:+,.2f}")
if last_ma:
    print(f"{ma_label:15s} ฿{last_ma:.2f}  (price {ma_distance:+.2f}% vs MA)")
else:
    print(f"{ma_label}")

# --- Diagnostic: what columns do we actually have? ---
print("\n--- Column check ---")
print("buys      :", df_buys_dash.columns.tolist())
print("sells     :", df_sells_dash.columns.tolist())
print("dividends :", df_div_dash.columns.tolist())
print("rcl       :", df_rcl_dash.columns.tolist())
print("set       :", df_set_dash.columns.tolist())

# =====================================================================
# SERIALIZE — each loop is separate and self-contained
# =====================================================================

# ---- BUYS ----
buys_js = []
for _, r in df_buys_dash.iterrows():
    buys_js.append({
        "status": str(r["status"]),
        "date":   str(r["date"]),
        "qty":    int(r["qty"]),
        "price":  float(r["price"]),
        "net":    float(r["net"]),
        "kind":   str(r["kind"]) if "kind" in r.index else "",
        "id":     int(r["id"]),
    })

# ---- SELLS ----
sells_js = []
for _, r in df_sells_dash.iterrows():
    sells_js.append({
        "sellDate": str(r["sell_date"]) if "sell_date" in r.index else "",
        "buyDate":  str(r["buy_date"])  if "buy_date"  in r.index else "",
        "days":     int(r["days"])      if "days"      in r.index else 0,
        "qty":      int(r["qty"])       if "qty"       in r.index else 0,
        "cost":     float(r["cost"])    if "cost"      in r.index else 0.0,
        "proceeds": float(r["proceeds"])if "proceeds"  in r.index else 0.0,
        "profit":   float(r["profit"])  if "profit"    in r.index else 0.0,
        "pct":      float(r["percent"]) if "percent"   in r.index else 0.0,
        "kind":     str(r["kind"])      if "kind"      in r.index else "",
    })

# ---- DIVIDENDS ----
divs_js = []
for _, r in df_div_dash.iterrows():
    divs_js.append({
        "year": int(r["year"]),
        "net":  float(r["net"]),
    })

# ---- PRICE + MA SERIES ----
rcl_js = []
for i, (_, r) in enumerate(df_rcl_dash.iterrows()):
    rcl_js.append({
        "date":  str(r["date"]),
        "price": float(r["price"]),
        "ma":    ma_series[i] if i < len(ma_series) else None,
    })

# ---- SET INDEX ----
set_js = []
for _, r in df_set_dash.iterrows():
    set_js.append({
        "date":  str(r["date"]),
        "value": float(r["setindex"]),
    })

# ---- ASSEMBLE ----
payload = {
    "name":          NAME,
    "latestPrice":   LATEST_PRICE,
    "latestDate":    LATEST_DATE,
    "lastMA":        last_ma if last_ma else 0,
    "maWindow":      MA_WINDOW if MA_WINDOW else 0,
    "maLabel":       ma_label,
    "maDistance":    ma_distance,
    "activeCost":    active_cost,
    "activeQty":     active_qty,
    "marketValue":   market_value,
    "unrealized":    unrealized,
    "unrealizedPct": unrealized_pct,
    "realized":      realized_profit,
    "dividends":     total_dividends,
    "totalReturn":   total_return,
    "buys":          buys_js,
    "sells":         sells_js,
    "dividendsList": divs_js,
    "rcl":           rcl_js,
    "set":           set_js,
    "buyZones":       buy_zones,
    "buyZoneStatus":  buy_zone_status,
    "maTrend":        ma_trend,
    "maTrendColor":   ma_trend_color,
    "maTrend20dPct":  ma_trend_20d_pct,
    "maTrend60dPct":  ma_trend_60d_pct,
}

payload_json = json.dumps(payload, ensure_ascii=False)
print(f"\n✅ Payload built — {len(payload_json):,} chars")
print(f"   buys:   {len(buys_js)}")
print(f"   sells:  {len(sells_js)}")
print(f"   divs:   {len(divs_js)}")
print(f"   rcl:    {len(rcl_js)}")
print(f"   set:    {len(set_js)}")
print(f"   MA window: {MA_WINDOW if MA_WINDOW else 'none'}")


--- 200-day MA direction ---
  MA now       : ฿1.81
  MA 20d ago   : ฿1.83  (-1.22%)
  MA 60d ago   : ฿1.92  (-5.99%)
  Trend        : FALLING

--- Technical buy zones (200-day MA) ---
  Trend-following     ฿    1.81 – ฿    1.90   Buy on pullback to just above the 200-MA
  Mean-reversion      ฿    1.63 – ฿    1.81   Buy when price dips below the 200-MA and turns back up
  Deep value          ฿    1.36 – ฿    1.63   Aggressive buy only if the thesis is intact
  Today (฿1.78): INSIDE — Mean-reversion zone

--- Portfolio summary ---
Active cost:    ฿521,151.75
Active qty:     50,000
Market value:   ฿89,000.00
Unrealized P&L: ฿-432,151.75 (-82.92%)
Realized:       ฿-28,109.91
Dividends:      ฿152,640.00
Total return:   ฿-307,621.66
200-day MA      ฿1.81  (price -1.64% vs MA)

--- Column check ---
buys      : ['id', 'stock_id', 'date', 'qty', 'price', 'fee', 'vat', 'status', 'net', 'kind', 'name']
sells     : ['id', 'buy_id', 'sell_date', 'sell_price', 'proceeds', 'days', 'profit', 'percen

In [7]:
# =====================================================================
# CELL 07 — Template: head + CSS
# Builds the <head> and top-of-body markup. Ends with an open .grid.
# Concatenated into `dashboard_html` by Cell 13.
# =====================================================================

part_head = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>{name} Portfolio Dashboard (THB)</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<style>
  :root {{
    --bg:#0f1720; --panel:#17212b; --text:#e6edf3; --muted:#8b9aa8;
    --accent:#4fc3f7; --green:#4caf50; --red:#ef5350; --amber:#ffb74d; --border:#26333f;
  }}
  * {{ box-sizing: border-box; }}
  body {{ margin:0; background:var(--bg); color:var(--text);
          font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif; padding:24px; }}
  h1 {{ font-size:24px; margin:0 0 4px 0; font-weight:600; }}
  .sub {{ color:var(--muted); font-size:13px; margin-bottom:24px; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr));
           gap:12px; margin-bottom:24px; }}
  .kpi {{ background:var(--panel); border:1px solid var(--border);
          border-radius:10px; padding:14px 16px; }}
  .kpi .label {{ color:var(--muted); font-size:11px; text-transform:uppercase;
                 letter-spacing:0.5px; margin-bottom:6px; }}
  .kpi .value {{ font-size:22px; font-weight:600; }}
  .kpi .sub-value {{ font-size:12px; color:var(--muted); margin-top:4px; }}
  .grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(420px,1fr)); gap:16px; }}
  .card {{ background:var(--panel); border:1px solid var(--border);
           border-radius:10px; padding:16px; }}
  .card h2 {{ font-size:14px; margin:0 0 12px 0; font-weight:600; }}
  .card .hint {{ font-size:11px; color:var(--muted); margin-top:10px; line-height:1.5; }}
  .chart-wrap {{ position:relative; height:260px; }}
  .chart-wrap.tall {{ height:320px; }}
  .green {{ color:var(--green); }} .red {{ color:var(--red); }} .amber {{ color:var(--amber); }}
  table.data {{ width:100%; border-collapse:collapse; font-size:12px; }}
  table.data th, table.data td {{ text-align:left; padding:6px 8px; border-bottom:1px solid var(--border); }}
  table.data th {{ color:var(--muted); font-weight:500; text-transform:uppercase;
                   font-size:10px; letter-spacing:0.5px; }}
  table.data td.num {{ text-align:right; font-variant-numeric:tabular-nums; }}
</style>
</head>
<body>

<h1>{name} Portfolio Dashboard</h1>
<div class="sub" id="subline">Loading…</div>

<div class="kpis" id="kpis"></div>

<div class="grid">
"""

In [8]:
# =====================================================================
# CELL 08 — Template: body + cards
# The four top-of-grid cards, the two full-width tables, the
# buy-zone chart card, the sells table, and the dividend bar card.
# Ends by closing .grid, leaving <script> for Cell 09.
# =====================================================================

part_body = """  <div class="card">
    <h2>Total Return Composition (฿)</h2>
    <div class="chart-wrap"><canvas id="composeChart"></canvas></div>
    <div class="hint">Dividends vs realized vs unrealized.</div>
  </div>

  <div class="card">
    <h2>Return by Year (Realized + Dividends)</h2>
    <div class="chart-wrap"><canvas id="yearChart"></canvas></div>
    <div class="hint">Stacked: capital gains + net dividends.</div>
  </div>

  <div class="card">
    <h2>{name} Price + 200-Day MA</h2>
    <div class="chart-wrap tall"><canvas id="maChart"></canvas></div>
    <div class="hint">Above MA = bullish regime.</div>
  </div>

  <div class="card">
    <h2>{name} vs SET Index (Rebased to 100)</h2>
    <div class="chart-wrap tall"><canvas id="vsChart"></canvas></div>
    <div class="hint">Is {name} outperforming the Thai market?</div>
  </div>

  <div class="card" style="grid-column:1 / -1;">
    <h2>Active Lots vs 200-Day MA — Sell Signal Table</h2>
    <table class="data" id="signalTable">
      <thead><tr>
        <th>ID</th><th>Buy Date</th><th class="num">Qty</th><th class="num">Cost/sh</th>
        <th class="num">Cost Basis</th><th class="num">Market Value</th>
        <th class="num">P/L</th><th class="num">P/L %</th><th class="num">vs MA</th><th>Signal</th>
      </tr></thead>
      <tbody></tbody>
    </table>
  </div>

  <div class="card" style="grid-column:1 / -1;">
    <h2>Technical Buy Zones — Reference Table</h2>
    <table class="data" id="buyZoneTable">
      <thead><tr>
        <th>Zone</th><th class="num">From (฿)</th><th class="num">To (฿)</th>
        <th class="num">Distance from today</th>
        <th>Trend context</th><th>Rationale</th>
      </tr></thead>
      <tbody></tbody>
    </table>
    <div class="hint">
      Zones are derived from the 200-day MA × multipliers (1.00–1.05, 0.90–1.00, 0.75–0.90).
      These are technical levels only — they don't assess whether the stock is fundamentally cheap.
    </div>
  </div>

  <div class="card">
    <h2>Technical Buy Zones vs 200-MA</h2>
    <div class="chart-wrap tall"><canvas id="buyZoneChart"></canvas></div>
    <div class="hint" id="buyZoneHint"></div>
  </div>

  <div class="card" style="grid-column:1 / -1;">
    <h2>Closed Trades Detail (฿)</h2>
    <table class="data" id="sellsTable">
      <thead><tr>
        <th>Sell Date</th><th>Buy Date</th><th class="num">Days</th>
        <th class="num">Qty</th><th class="num">Cost</th><th class="num">Proceeds</th>
        <th class="num">Profit</th><th class="num">Return</th><th class="num">Annualized</th>
      </tr></thead>
      <tbody></tbody>
    </table>
  </div>

  <div class="card" style="grid-column:1 / -1;">
    <h2>Dividend Income by Year (฿ net)</h2>
    <div class="chart-wrap"><canvas id="divYearChart"></canvas></div>
    <div class="hint">Net dividends per calendar year, based on <code>df_div_dash</code>. Bars stack per year; multiple payments in the same year are summed.</div>
  </div>

</div>
"""

In [9]:
# =====================================================================
# CELL 09 — JS: helpers + KPIs + top charts
# The `P = {payload}` constant, THB formatters, the annualizedPct
# helper, and the four charts in the top grid: composeChart,
# yearChart, maChart, vsChart.
# =====================================================================

part_js_helpers_topcharts = """
<script>
const P = {payload};
const THB  = n => '฿' + Number(n).toLocaleString('en-US', {{maximumFractionDigits:0}});
const THB2 = n => '฿' + Number(n).toLocaleString('en-US', {{minimumFractionDigits:2,maximumFractionDigits:2}});
const fmt  = n => Number(n).toLocaleString('en-US', {{maximumFractionDigits:0}});

function annualizedPct(dec, days) {{
  if (days <= 0) return 0;
  return (Math.pow(1 + dec, 365/days) - 1) * 100;
}}

Chart.defaults.color = '#8b9aa8';
Chart.defaults.borderColor = '#26333f';

document.getElementById('subline').innerHTML =
  `Snapshot ${{P.latestDate}} · ${{P.name}} ฿${{P.latestPrice.toFixed(2)}} · 200-MA ฿${{P.lastMA.toFixed(2)}} · All amounts in Thai Baht (฿)`;

// ---- KPIs ----
const kpis = [
    {{label:'Buy Zone Signal', value: P.buyZoneStatus.split(' — ')[0],
     sub:'200-MA ฿' + P.lastMA.toFixed(2),
     cls: P.buyZoneStatus.includes('INSIDE') ? 'green'
        : P.buyZoneStatus.includes('BELOW')  ? 'amber' : 'red'}},
    {{label:'200-MA Trend', value: P.maTrend,
      sub: (P.maTrend20dPct>=0?'+':'') + P.maTrend20dPct.toFixed(2) + '% over 20d',
      cls: P.maTrend === 'RISING' ? 'green'
         : P.maTrend === 'FALLING' ? 'red' : 'amber'}},
  {{label:'Active Position', value:THB(P.marketValue),
    sub: fmt(P.activeQty) + ' sh @ ฿' + P.latestPrice.toFixed(2)}},
  {{label:'Active Cost Basis', value:THB(P.activeCost),
    sub:'avg ฿' + (P.activeCost/P.activeQty).toFixed(2) + '/sh'}},
  {{label:'Unrealized P&L', value:(P.unrealized>=0?'+':'−')+THB(Math.abs(P.unrealized)),
    sub:(P.unrealizedPct>=0?'+':'')+P.unrealizedPct.toFixed(2)+'%',
    cls: P.unrealized>=0?'green':'red'}},
  {{label:'Realized Gains', value:(P.realized>=0?'+':'−')+THB(Math.abs(P.realized)),
    sub:P.sells.length + ' closed trades',
    cls: P.realized>=0?'green':'red'}},
  {{label:'Net Dividends', value:THB(P.dividends),
    sub:P.dividendsList.length + ' payments', cls:'green'}},
  {{label:'Total Return',
    value:(P.totalReturn>=0?'+':'−')+THB(Math.abs(P.totalReturn)),
    sub:'realized + dividends + unrealized',
    cls: P.totalReturn>=0?'green':'red'}},
  {{label:'Price vs 200-MA', value:'฿' + P.latestPrice.toFixed(2),
    sub:(P.maDistance>=0?'+':'')+P.maDistance.toFixed(2)+'% vs ฿'+P.lastMA.toFixed(2),
    cls: P.maDistance>=0?'green':'red'}},
];
document.getElementById('kpis').innerHTML = kpis.map(k =>
  `<div class="kpi"><div class="label">${{k.label}}</div>
   <div class="value ${{k.cls||''}}">${{k.value}}</div>
   <div class="sub-value">${{k.sub}}</div></div>`).join('');

// ---- 1. Composition ----
new Chart(document.getElementById('composeChart'), {{
  type:'doughnut',
  data:{{ labels:['Net Dividends','Realized Gains','Unrealized P&L'],
    datasets:[{{ data:[P.dividends, P.realized, Math.max(P.unrealized,0)],
      backgroundColor:['#4fc3f7','#4caf50','#ffb74d'],
      borderColor:'#17212b', borderWidth:3 }}] }},
  options:{{responsive:true, maintainAspectRatio:false,
    plugins:{{legend:{{position:'bottom', labels:{{padding:14, boxWidth:12}}}},
             tooltip:{{callbacks:{{label:c => c.label + ': ' + THB(c.raw)}}}}}}}}
}});

// ---- 2. Year stack ----
const years = [...new Set([...P.dividendsList.map(d=>d.year),
                            ...P.sells.map(s=>+s.sellDate.slice(0,4))])].sort();
const divByYear  = years.map(y => P.dividendsList.filter(d=>d.year===y).reduce((s,d)=>s+d.net,0));
const gainByYear = years.map(y => P.sells.filter(s=>+s.sellDate.slice(0,4)===y).reduce((s,x)=>s+x.profit,0));
new Chart(document.getElementById('yearChart'), {{
  type:'bar',
  data:{{labels:years, datasets:[
    {{label:'Net Dividends', data:divByYear, backgroundColor:'#4fc3f7'}},
    {{label:'Realized Gains', data:gainByYear, backgroundColor:'#4caf50'}} ]}},
  options:{{responsive:true, maintainAspectRatio:false,
    scales:{{x:{{stacked:true, grid:{{display:false}}}},
            y:{{stacked:true, ticks:{{callback:v => '฿' + (v/1000).toFixed(0) + 'k'}}}}}},
    plugins:{{legend:{{position:'bottom', labels:{{padding:12, boxWidth:12}}}},
             tooltip:{{callbacks:{{label:c => c.dataset.label + ': ' + THB(c.raw)}}}}}}}}
}});

// ---- 3. MA chart ----
new Chart(document.getElementById('maChart'), {{
  type:'line',
  data:{{labels:P.rcl.map(p=>p.date), datasets:[
    {{label:'Close', data:P.rcl.map(p=>p.price), borderColor:'#4fc3f7',
      backgroundColor:'rgba(79,195,247,0.08)', fill:true, tension:0.15,
      pointRadius:0, borderWidth:1.5}},
    {{label:'200-Day MA', data:P.rcl.map(p=>p.ma), borderColor:'#ffb74d',
      borderWidth:2, pointRadius:0, tension:0.15, spanGaps:true}} ]}},
  options:{{responsive:true, maintainAspectRatio:false,
    interaction:{{mode:'index', intersect:false}},
    scales:{{x:{{ticks:{{maxRotation:0, autoSkip:true, maxTicksLimit:8}}}},
            y:{{ticks:{{callback:v => '฿' + v}}}}}},
    plugins:{{legend:{{position:'bottom', labels:{{padding:12, boxWidth:12}}}},
             tooltip:{{callbacks:{{label:c => c.dataset.label + ': ฿' + (c.raw!=null ? c.raw.toFixed(2) : '—')}}}}}}}}
}});
"""

# Append to accumulating script part
part_js = part_js_helpers_topcharts
print(f"✅ Cell 09 — helpers + top charts — {len(part_js):,} chars")

✅ Cell 09 — helpers + top charts — 5,030 chars


In [10]:
# =====================================================================
# CELL 10 — JS: buy-zone overlay + vs SET
# The Plotly overlay that draws the shaded buy-zone bands on top of
# price + 200-MA, and the Chart.js line comparing {name} against the
# SET index (rebased to 100).
# =====================================================================

part_js_buyzone_vsset = """
// ---- Buy zones overlay on price + MA ----
(function(){{
  const zones = P.buyZones;
  const lastMA = P.lastMA;

  const xStart = P.rcl[0].date;
  const xEnd   = P.rcl[P.rcl.length - 1].date;

  const shapes = zones.map(z => ({{
    type: 'rect',
    xref: 'x', yref: 'y',
    x0: xStart, x1: xEnd,
    y0: z.lo,   y1: z.hi,
    fillcolor: z.color,
    opacity: 0.12,
    line: {{ width: 0 }},
    layer: 'below',
  }}));

  shapes.push({{
    type: 'line',
    xref: 'paper', yref: 'y',
    x0: 0, x1: 1,
    y0: lastMA, y1: lastMA,
    line: {{ color: '#ffb74d', width: 1.5, dash: 'dot' }},
  }});

  shapes.push({{
    type: 'line',
    xref: 'paper', yref: 'y',
    x0: 0, x1: 1,
    y0: P.latestPrice, y1: P.latestPrice,
    line: {{ color: '#4fc3f7', width: 1.5, dash: 'dash' }},
  }});

  const trace1 = {{
    x: P.rcl.map(p => p.date),
    y: P.rcl.map(p => p.price),
    mode: 'lines',
    name: 'Close',
    line: {{ color: '#e6edf3', width: 1.5 }},
    hovertemplate: '%{{x}}<br>฿%{{y:.2f}}<extra></extra>',
  }};

  const trace2 = {{
    x: P.rcl.map(p => p.date),
    y: P.rcl.map(p => p.ma),
    mode: 'lines',
    name: '200-MA',
    line: {{ color: '#ffb74d', width: 2 }},
    connectgaps: false,
    hovertemplate: '%{{x}}<br>MA ฿%{{y:.2f}}<extra></extra>',
  }};

  const maDirectionText = P.maTrend === 'RISING'   ? '↗ MA RISING'
                         : P.maTrend === 'FALLING' ? '↘ MA FALLING'
                         :                           '→ MA FLAT';

  const annotations = zones.map(z => ({{
    x: xEnd, y: (z.lo + z.hi) / 2,
    xref: 'x', yref: 'y',
    text: `${{z.label}}<br>฿${{z.lo}}–฿${{z.hi}}`,
    showarrow: false,
    xanchor: 'right',
    font: {{ size: 10, color: z.color }},
  }})).concat([{{
    x: xEnd,
    y: P.lastMA,
    xref: 'x', yref: 'y',
    text: maDirectionText,
    showarrow: false,
    xanchor: 'right',
    yanchor: 'bottom',
    font: {{ size: 11, color: P.maTrendColor, weight: 'bold' }},
  }}]);

  Plotly.newPlot('buyZoneChart', [trace1, trace2], {{
    margin: {{ t: 20, b: 40, l: 55, r: 90 }},
    shapes,
    annotations,
    xaxis: {{ showgrid: false, tickfont: {{ size: 10 }} }},
    yaxis: {{ title: 'Price (฿)', tickprefix: '฿' }},
    legend: {{ orientation: 'h', y: 1.05 }},
    plot_bgcolor: '#17212b',
    paper_bgcolor: '#17212b',
    font: {{ color: '#8b9aa8' }},
    hovermode: 'x unified',
  }}, {{ responsive: true, displayModeBar: false }});

  document.getElementById('buyZoneHint').innerHTML =
    `<b>Today:</b> ฿${{P.latestPrice.toFixed(2)}} — ${{P.buyZoneStatus}}.<br>` +
    zones.map(z => `&nbsp;&nbsp;<span style="color:${{z.color}}">■</span> ${{z.label}}: ฿${{z.lo}}–฿${{z.hi}} — ${{z.sub}}`).join('<br>');
}})();

// ---- 4. vs SET ----
const setMap = new Map(P.set.map(p => [p.date, p.value]));
const aligned = P.rcl.filter(p => setMap.has(p.date));
if (aligned.length > 0) {{
  const baseRCL = aligned[0].price;
  const baseSET = setMap.get(aligned[0].date);
  const rclN = aligned.map(p => +(p.price/baseRCL*100).toFixed(2));
  const setN = aligned.map(p => +(setMap.get(p.date)/baseSET*100).toFixed(2));
  new Chart(document.getElementById('vsChart'), {{
    type:'line',
    data:{{labels:aligned.map(p=>p.date), datasets:[
      {{label:P.name+' (rebased)', data:rclN, borderColor:'#4fc3f7', borderWidth:2, pointRadius:0, tension:0.15}},
      {{label:'SET Index (rebased)', data:setN, borderColor:'#ffb74d', borderWidth:2, pointRadius:0, tension:0.15}} ]}},
    options:{{responsive:true, maintainAspectRatio:false,
      interaction:{{mode:'index', intersect:false}},
      scales:{{x:{{ticks:{{maxRotation:0, autoSkip:true, maxTicksLimit:8}}}}}},
      plugins:{{legend:{{position:'bottom', labels:{{padding:12, boxWidth:12}}}}}}}}
  }});
}}
"""

part_js += part_js_buyzone_vsset
print(f"✅ Cell 10 — buy-zone + vs-SET — running total {len(part_js):,} chars")

✅ Cell 10 — buy-zone + vs-SET — running total 8,781 chars


In [11]:
# =====================================================================
# CELL 11 — JS: dividends by year
# Aggregates P.dividendsList (one entry per payment) into one bar per
# calendar year, then renders into #divYearChart.
# =====================================================================

part_js_dividends = """
// ---- Dividends by Year ----
const divYears = [...new Set(P.dividendsList.map(d => d.year))].sort((a,b) => a-b);
const divByYearMap = new Map();
divYears.forEach(y => divByYearMap.set(y, 0));
P.dividendsList.forEach(d => divByYearMap.set(d.year, divByYearMap.get(d.year) + d.net));
const divByYearValues = divYears.map(y => +divByYearMap.get(y).toFixed(2));
const divTotalAll = divByYearValues.reduce((a,b) => a+b, 0);
new Chart(document.getElementById('divYearChart'), {{
  type:'bar',
  data:{{labels:divYears, datasets:[{{
    label:'Net Dividends (฿)',
    data:divByYearValues,
    backgroundColor:'#4fc3f7',
    borderColor:'#4fc3f7', borderWidth:1
  }}]}},
  options:{{responsive:true, maintainAspectRatio:false,
    scales:{{x:{{grid:{{display:false}}, ticks:{{maxRotation:0, autoSkip:false}}}},
            y:{{ticks:{{callback:v => '฿' + Number(v).toLocaleString('en-US',{{maximumFractionDigits:0}})}}}}}},
    plugins:{{legend:{{display:false}},
             tooltip:{{callbacks:{{
               label:c => 'Net: ' + THB(c.raw),
               afterLabel:c => 'Share of total: ' + (divTotalAll ? (c.raw/divTotalAll*100).toFixed(1) : '0.0') + '%'
             }}}}}},
  }}
}});
"""

part_js += part_js_dividends
print(f"✅ Cell 11 — dividends-by-year — running total {len(part_js):,} chars")

✅ Cell 11 — dividends-by-year — running total 9,973 chars


In [12]:
# =====================================================================
# CELL 12 — JS: tables + close
# Renders the three full-width tables (Active Lots, Closed Trades,
# Buy Zones reference) and closes the <script> and </html>.
# =====================================================================

part_js_tables_close = """
// ---- 7. Signal table ----
const maDelta = P.maDistance;
document.querySelector('#signalTable tbody').innerHTML =
  P.buys.filter(b => b.status === 'Active').map(b => {{
    const mv = b.qty * P.latestPrice;
    const pl = mv - b.net;
    const plPct = pl / b.net * 100;
    let signal, cls;
    if (P.latestPrice > P.lastMA && plPct > 15) {{ signal = 'TAKE PROFIT'; cls = 'green'; }}
    else if (P.latestPrice > P.lastMA)          {{ signal = 'HOLD · TREND UP'; cls = 'green'; }}
    else if (plPct < -10)                        {{ signal = 'REASSESS'; cls = 'red'; }}
    else                                          {{ signal = 'HOLD'; cls = 'amber'; }}
    return `<tr>
      <td>${{b.id}}</td><td>${{b.date}}</td>
      <td class="num">${{fmt(b.qty)}}</td>
      <td class="num">฿${{b.price.toFixed(2)}}</td>
      <td class="num">${{THB(b.net)}}</td>
      <td class="num">${{THB(mv)}}</td>
      <td class="num ${{pl>=0?'green':'red'}}">${{pl>=0?'+':'−'}}${{THB(Math.abs(pl))}}</td>
      <td class="num ${{plPct>=0?'green':'red'}}">${{plPct>=0?'+':''}}${{plPct.toFixed(1)}}%</td>
      <td class="num ${{maDelta>=0?'green':'red'}}">${{maDelta>=0?'+':''}}${{maDelta.toFixed(1)}}%</td>
      <td class="${{cls}}">${{signal}}</td>
    </tr>`;
  }}).join('');

// ---- 8. Sells table ----
document.querySelector('#sellsTable tbody').innerHTML = P.sells.map(s => {{
  const ann = annualizedPct(s.pct/100, s.days);
  const annDisplay = ann > 999 ? '>999%' : ann.toFixed(1) + '%';
  return `<tr>
    <td>${{s.sellDate}}</td><td>${{s.buyDate}}</td>
    <td class="num">${{s.days}}</td>
    <td class="num">${{fmt(s.qty)}}</td>
    <td class="num">${{THB2(s.cost)}}</td>
    <td class="num">${{THB2(s.proceeds)}}</td>
    <td class="num ${{s.profit>=0?'green':'red'}}">${{s.profit>=0?'+':'−'}}${{THB2(Math.abs(s.profit))}}</td>
    <td class="num ${{s.pct>=20?'green':s.pct>=10?'':'amber'}}">${{s.pct.toFixed(2)}}%</td>
    <td class="num ${{ann>=50?'green':ann>=15?'amber':'red'}}">${{annDisplay}}</td>
  </tr>`;
}}).join('');

document.querySelector('#buyZoneTable tbody').innerHTML =
  P.buyZones.map(z => {{
    const mid  = (z.lo + z.hi) / 2;
    const dist = ((P.latestPrice - mid) / mid * 100);
    const distCls = Math.abs(dist) <= 5 ? 'green' : dist > 0 ? 'amber' : 'green';
    const distLabel = dist > 0
      ? `+${{dist.toFixed(1)}}% above`
      : `${{dist.toFixed(1)}}% below`;

    let trendNote = '';
    if (z.label === 'Trend-following') {{
      trendNote = P.maTrend === 'RISING'
        ? '✅ MA rising — pullbacks are buyable'
        : P.maTrend === 'FALLING'
          ? '⚠️ MA falling — wait for a reclaim'
          : '➖ MA flat — wait for direction';
    }} else if (z.label === 'Mean-reversion') {{
      trendNote = P.maTrend === 'RISING'
        ? '✅ Healthy dip in an uptrend'
        : P.maTrend === 'FALLING'
          ? '⚠️ Catching a falling knife — size small'
          : '➖ Range-bound — normal accumulation';
    }} else {{
      trendNote = P.maTrend === 'FALLING'
        ? '⚠️ Only if thesis is intact'
        : '✅ Best entry if thesis holds';
    }}

    return `<tr>
      <td style="color:${{z.color}}; font-weight:600">${{z.label}}</td>
      <td class="num">฿${{z.lo.toFixed(2)}}</td>
      <td class="num">฿${{z.hi.toFixed(2)}}</td>
      <td class="num ${{distCls}}">${{distLabel}}</td>
      <td style="color:${{P.maTrendColor}}">${{trendNote}}</td>
      <td>${{z.note}}</td>
    </tr>`;
  }}).join('');

</script>
</body>
</html>
"""

part_js += part_js_tables_close
print(f"✅ Cell 12 — tables + close — running total {len(part_js):,} chars")

✅ Cell 12 — tables + close — running total 13,460 chars


In [13]:
# =====================================================================
# CELL 13 — Assemble template + write HTML
# Concatenates the four string fragments from Cells 07-12 into one
# `dashboard_html` template, substitutes {name} + {payload}, and
# writes the HTML file next to the CSVs.
#
# This cell must run AFTER Cells 07-12 have populated `part_head`,
# `part_body`, and `part_js`. It must run BEFORE Cell 14, which
# reads the file back to inject the drop→bounce appendix.
# =====================================================================

dashboard_html = part_head + part_body + part_js

rendered = dashboard_html.format(name=NAME, payload=payload_json)

output_file = os.path.join(csv_path, f"{NAME}-dashboard-{dd}.html")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(rendered)

print(f"✅ Dashboard generated: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024:.2f} KB")

✅ Dashboard generated: C:\Users\PC1\OneDrive\A4\CSV\ORI-dashboard-28.html
File size: 93.99 KB


In [14]:
# =====================================================================
# BOUNCE→DROP BASE-RATE APPENDIX  (replaces Drop→Bounce)
# =====================================================================
# STRATEGY
#   You already hold N shares of {NAME}. On a ≥5% up-day you SELL at
#   the close (booking a spike). You then wait to buy back ≥5% below
#   that sell price, restoring your share count. The profit per share
#   is (sell_price − buy_price), i.e. the "spread" the bounce
#   handed you — without changing your long-term position size.
#
#   This cell scans the FULL price history the dashboard already
#   loaded, measures how often a ≥5% up-day is followed by a ≥5%
#   drop-back within 5/10/20 trading rows, and injects two cards
#   into the HTML:
#     1. the base-rate table + KPI tiles
#     2. a collapsible 'Method & rationale' block
#   It also writes {NAME}-sim-notes-short.json next to the HTML so
#   every threshold is auditable.
#
# HOW TO FINE-TUNE
#   Edit the tunable constants below, re-run cell 7 (to rebuild the
#   base dashboard), then re-run this cell. The cell is idempotent:
#   it strips its own previous appendix before re-inserting.
# =====================================================================

import json, os, re
from datetime import datetime

# ---------- tunable constants (mirrored into the JSON sidecar) ----------
RISE_PCT        = 0.05      # qualifying up-day size
TARGET_PCT      = 0.05      # buy-back target below the sell price
HORIZONS_ROWS   = (5, 10, 20)
MA_GATE_WINDOW  = 200

# ---------- reuse the exact series the dashboard already built ----------
px = (
    df_rcl_dash[["date", "price"]]
    .copy()
    .assign(price=lambda d: d["price"].astype(float))
    .reset_index(drop=True)
)
px["ma"] = px["price"].rolling(MA_GATE_WINDOW, min_periods=MA_GATE_WINDOW).mean()

# ---------- find qualifying UP-days (prior close -> today) ----------
px["prior_close"] = px["price"].shift(1)
px["rise_pct"]    = (px["price"] / px["prior_close"] - 1.0) * 100.0
candidates = px[(px["prior_close"].notna()) & (px["rise_pct"] >= RISE_PCT * 100)].copy()

# ---------- walk forward for each candidate ----------
def _first_drop(idx_entry: int, entry_px: float):
    """Return (row_offset or None, drop_close, rows_to_target, max_runup_pct_before_hit)."""
    target  = entry_px * (1 - TARGET_PCT)
    max_runup = 0.0
    for k in range(1, max(HORIZONS_ROWS) + 1):
        j = idx_entry + k
        if j >= len(px):
            return None, None, None, max_runup
        c = float(px.at[j, "price"])
        runup = (c - entry_px) / entry_px * 100.0
        if runup > max_runup:
            max_runup = runup
        if c <= target:
            return k, c, j - idx_entry, max_runup
    return None, None, None, max_runup

rows = []
for _, r in candidates.iterrows():
    i     = int(r.name)
    entry = float(r["price"])
    hit_k, hit_close, rows_to_hit, max_runup = _first_drop(i, entry)

    cal_days = None
    if hit_k is not None:
        d0 = datetime.fromisoformat(str(px.at[i, "date"]))
        d1 = datetime.fromisoformat(str(px.at[i + hit_k, "date"]))
        cal_days = (d1 - d0).days

    # Spread realized = (sell − buy_back) / sell. Positive when we bought back lower.
    spread_pct = None
    if hit_close is not None and entry:
        spread_pct = (entry - hit_close) / entry * 100.0

    above_ma = bool(pd.notna(r["ma"]) and entry > r["ma"])

    rows.append({
        "up_date":         str(r["date"]),
        "entry_close":     round(entry, 4),
        "prior_close":     round(float(r["prior_close"]), 4),
        "rise_pct":        round(float(r["rise_pct"]), 2),
        "target_price":    round(entry * (1 - TARGET_PCT), 4),
        "hit_date":        str(px.at[i + hit_k, "date"]) if hit_k else "",
        "rows_to_target":  rows_to_hit if hit_k else None,
        "cal_days":        cal_days,
        "spread_pct":      round(spread_pct, 2) if spread_pct is not None else None,
        "max_runup_before_hit_pct": round(max_runup, 2),
        "above_200ma_at_entry": above_ma,
    })

ep = pd.DataFrame(rows)

# ---------- aggregate ----------
def _hit_rate(df, h):
    if df.empty: return 0.0
    return float((df["rows_to_target"].notna() & (df["rows_to_target"] <= h)).mean()) * 100.0

n_total    = len(ep)
hit5       = _hit_rate(ep, 5)
hit10      = _hit_rate(ep, 10)
hit20      = _hit_rate(ep, 20)
median_k   = int(ep.loc[ep["rows_to_target"].notna(), "rows_to_target"].median()) if n_total else 0
worst_up   = float(ep["max_runup_before_hit_pct"].max()) if n_total else 0.0

n_hit_target = int(ep["rows_to_target"].notna().sum())
avg_spread_on_hits = (
    round(ep.loc[ep["spread_pct"].notna(), "spread_pct"].mean(), 2)
    if ep["spread_pct"].notna().any() else 0.0
)

ep_above = ep[ep["above_200ma_at_entry"]]
ep_below = ep[~ep["above_200ma_at_entry"]]
hit20_above = _hit_rate(ep_above, 20) if len(ep_above) else 0.0
hit20_below = _hit_rate(ep_below, 20) if len(ep_below) else 0.0

# ---------- build the section HTML ----------
def _fmt_pct(v):
    return "—" if v is None or (isinstance(v, float) and pd.isna(v)) else f"{v:+.2f}%"

table_rows = "\n".join(
    f"<tr>"
    f"<td>{r['up_date']}</td>"
    f"<td class='num'>฿{r['entry_close']:.2f}</td>"
    f"<td class='num {'green' if r['rise_pct']>0 else 'red'}'>{_fmt_pct(r['rise_pct'])}</td>"
    f"<td class='num'>฿{r['target_price']:.2f}</td>"
    f"<td>{r['hit_date'] or '—'}</td>"
    f"<td class='num'>{'—' if pd.isna(r['rows_to_target']) else int(r['rows_to_target'])}</td>"
    f"<td class='num'>{'—' if pd.isna(r['cal_days']) else int(r['cal_days'])}</td>"
    f"<td class='num {'green' if (r['spread_pct'] or 0)>0 else 'red'}'>"
    f"{'—' if pd.isna(r['spread_pct']) else format(r['spread_pct'], '.2f') + '%'}</td>"
    f"<td class='num {'red' if r['max_runup_before_hit_pct']>0 else ''}'>"
    f"{r['max_runup_before_hit_pct']:+.2f}%</td>"
    f"<td>{'above' if r['above_200ma_at_entry'] else 'below'}</td>"
    f"</tr>"
    for _, r in ep.iterrows()
)

section_html = f"""
<section class="card" style="grid-column:1 / -1; margin-top:24px;">
  <h2>Bounce→Drop Base Rate — “sell the spike, buy the dip” monitor</h2>
  <div class="hint" style="margin-bottom:14px;">
    Every close in the 3-yr file where <b>close ≥ prior_close × {1+RISE_PCT:.2f}</b>
    (a ≥{RISE_PCT*100:.0f}% <b>up</b> day). For each, we measure whether a
    <b>−{TARGET_PCT*100:.0f}%</b> buy-back target (below the sell price) was reached within
    <b>{', '.join(str(h) for h in HORIZONS_ROWS)}</b> trading rows.
    This is the historical answer to <i>“if I sell a {RISE_PCT*100:.0f}% spike,
    how often can I buy back {TARGET_PCT*100:.0f}% lower and pocket the spread?”</i>
  </div>

  <div class="kpis" style="margin-bottom:18px;">
    <div class="kpi"><div class="label">Qualifying up-days</div>
      <div class="value">{n_total}</div>
      <div class="sub-value">over {len(px)} trading rows</div></div>
    <div class="kpi"><div class="label">Buy-back rate ≤5 rows</div>
      <div class="value {'green' if hit5>=60 else 'amber'}">{hit5:.0f}%</div>
      <div class="sub-value">tightest window</div></div>
    <div class="kpi"><div class="label">Buy-back rate ≤10 rows</div>
      <div class="value {'green' if hit10>=70 else 'amber'}">{hit10:.0f}%</div>
      <div class="sub-value">median time-to-target {median_k} rows</div></div>
    <div class="kpi"><div class="label">Buy-back rate ≤20 rows</div>
      <div class="value {'green' if hit20>=80 else 'amber'}">{hit20:.0f}%</div>
      <div class="sub-value">worst run-up before hit {_fmt_pct(worst_up)}</div></div>
    <div class="kpi"><div class="label">Avg spread (on hits)</div>
      <div class="value green">{avg_spread_on_hits:.2f}%</div>
      <div class="sub-value">{n_hit_target} of {n_total} reached target</div></div>
    <div class="kpi"><div class="label">200-MA gate</div>
      <div class="value">{hit20_above:.0f}% vs {hit20_below:.0f}%</div>
      <div class="sub-value">{len(ep_above)} above MA · {len(ep_below)} below MA</div></div>
  </div>

  <table class="data">
    <thead><tr>
      <th>Up date</th><th class="num">Sell ฿</th><th class="num">Rise %</th>
      <th class="num">Buy-back ฿</th><th>Hit date</th><th class="num">Rows</th>
      <th class="num">Cal days</th><th class="num">Spread %</th>
      <th class="num">Max run-up before hit</th><th>vs 200-MA</th>
    </tr></thead>
    <tbody>{table_rows}</tbody>
  </table>
  <div class="hint" style="margin-top:10px;">
    Spread % is the simple gain from the up-day close to the first close
    at or below the −{TARGET_PCT*100:.0f}% buy-back target. Because the target itself is
    −{TARGET_PCT*100:.0f}%, every hit row will show ≥{TARGET_PCT*100:.0f}%;
    the interesting variation is how far below the target the first qualifying
    close landed (gap-downs), and the <b>Max run-up before hit</b> column —
    that's how much unrealized upside you had to endure while waiting.
  </div>
</section>

<section class="card" style="grid-column:1 / -1; margin-top:16px;">
  <h2>Method &amp; rationale — how to read this (collapsible)</h2>
  <details>
    <summary class="hint" style="cursor:pointer;">Click to expand the background notes</summary>
    <div class="hint" style="line-height:1.6; margin-top:10px;">
      <p><b>Why this section exists.</b> The dashboard already answers
      “when a stock drops, how often does it bounce?” This card answers the
      <i>mirror</i> question: “when a stock spikes, how often does it give
      back the spike so I can buy back lower?” You already own the shares,
      so a spike is a chance to sell high and re-enter low <b>without
      changing your long-term share count</b>. The profit is the spread,
      not a new position.</p>

      <p><b>The strategy in one line.</b> Sell N shares on a ≥{RISE_PCT*100:.0f}% up-day
      at the close. Place a buy-back limit at −{TARGET_PCT*100:.0f}% below
      that price. If it fills within the horizon, your share count returns
      to N and you keep the spread. If it doesn't fill, you either wait
      longer or buy back at market — either way you still own the shares,
      you just haven't banked the spread.</p>

      <p><b>Why we measure the spread, not the “profit”.</b>
      Unlike the drop→bounce card, here your “entry” is a sell. The
      relevant return is <code>(sell − buy_back) / sell</code>, which is
      what you'd actually pocket per share. Because the buy-back target
      is exactly −{TARGET_PCT*100:.0f}%, every hit shows ≥{TARGET_PCT*100:.0f}%
      spread; the interesting part is the gap-down tail (bigger spreads) and
      the <b>Max run-up before hit</b> column (how much the price ran
      against you before giving back).</p>

      <p><b>Why the 200-MA gate matters here too.</b> Selling a spike
      <i>below</i> the 200-MA in a falling stock is riskier than selling a
      spike above the MA in a rising stock: the latter is a normal pullback,
      the former might be the start of a new leg down (good for the spread)
      <i>or</i> a dead-cat bounce in a bear trend (bad — you might not get a
      chance to buy back before the stock recovers past your sell price).
      The tile “200-MA gate: {hit20_above:.0f}% vs {hit20_below:.0f}%”
      shows which regime is favourable for your symbol.</p>

      <p><b>Why we do not recommend “concentrate”.</b> Hit rates in this
      sample are {hit5:.0f}%–{hit20:.0f}%. The wins are capped near
      {TARGET_PCT*100:.0f}% spread, and the misses (especially when the
      price keeps rising) leave you out of N shares at a higher price than
      you sold — which is a real cost if you're forced to buy back higher.
      Position size should reflect that asymmetry, not the nominal spread.</p>

      <p><b>Data caveats.</b> (1) The exported
      <code>3-yrs-sales-data.csv</code> contains duplicate rows; harmless
      for base-rate counts but dedupe on <code>date</code> before
      trusting absolute numbers. (2) Thai holiday gaps compress
      calendar-day counts; the table reports both <b>Rows</b> and
      <b>Cal days</b>. (3) The 200-MA is <code>rolling({MA_GATE_WINDOW})</code>
      with <code>min_periods={MA_GATE_WINDOW}</code>, so early up-days
      show <code>below</code> even when the trend looked up.</p>

      <p><b>What to fine-tune next.</b> Change <code>RISE_PCT</code>,
      <code>TARGET_PCT</code>, <code>HORIZONS_ROWS</code>, or
      <code>MA_GATE_WINDOW</code>. Suggested experiments:
      (a) tighten to <code>RISE_PCT=0.06</code> — fewer but sharper spikes;
      (b) loosen <code>TARGET_PCT=0.03</code> — buy-back rate jumps, smaller
      spread; (c) set <code>MA_GATE_WINDOW=50</code> for a faster trend
      filter. Each run overwrites <code>{NAME}-sim-notes-short.json</code>.</p>
    </div>
  </details>
</section>
"""

# ---------- inject just before </div> that closes the .grid ----------
html_path = os.path.join(csv_path, f"{NAME}-dashboard-{dd}.html")
with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

# idempotent: strip BOTH possible previous appendices (old and new)
html = re.sub(
    r"\n<section class=\"card\" style=\"grid-column:1 / -1; margin-top:24px;\">"
    r".*?(Method &amp; rationale).*?</section>\s*",
    "\n", html, flags=re.DOTALL,
)

anchor = "</div>\n\n<script>"
assert anchor in html, "dashboard layout anchor not found — cell 7 output changed?"
html = html.replace(anchor, section_html + anchor, 1)

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Short appendix injected → {html_path}")
print(f"   size: {os.path.getsize(html_path)/1024:.1f} KB")
print(f"   up-days: {n_total}  |  hit≤5: {hit5:.0f}%  hit≤10: {hit10:.0f}%  hit≤20: {hit20:.0f}%")
print(f"   MA gate: above {hit20_above:.0f}% ({len(ep_above)})  below {hit20_below:.0f}% ({len(ep_below)})")
print(f"   avg spread on hits: {avg_spread_on_hits:.2f}%  ({n_hit_target} of {n_total} hits)")

# ---------- sidecar JSON ----------
sidecar = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "stock": NAME,
    "strategy": "sell_into_up_day_then_buy_back_lower",
    "source_csv_rows": int(len(px)),
    "thresholds": {
        "RISE_PCT":       RISE_PCT,
        "TARGET_PCT":     TARGET_PCT,
        "HORIZONS_ROWS":  list(HORIZONS_ROWS),
        "MA_GATE_WINDOW": MA_GATE_WINDOW,
    },
    "summary": {
        "n_qualifying_up_days": n_total,
        "hit_rate_pct":         {"h5": hit5, "h10": hit10, "h20": hit20},
        "median_rows_to_target": median_k,
        "worst_runup_pct":       worst_up,
        "avg_spread_pct_on_hits": avg_spread_on_hits,
        "n_hit_target":           n_hit_target,
        "above_ma": {"n": len(ep_above), "hit20_pct": hit20_above},
        "below_ma": {"n": len(ep_below), "hit20_pct": hit20_below},
    },
    "episodes": ep.to_dict(orient="records"),
    "notes": [
        "spread_pct = (entry_close - buy_back_close) / entry_close * 100, i.e. profit per share",
        "rows_to_target = trading rows after the up-day, not calendar days",
        "above_200ma_at_entry uses rolling(N, min_periods=N); first N-1 rows are NaN → 'below'",
        "source CSV contains exact-duplicate rows; dedupe on date before absolute counts",
        "This is the mirror of the Drop→Bounce card — same math, opposite sign",
    ],
}

sidecar_path = os.path.join(csv_path, f"{NAME}-sim-notes-short.json")
with open(sidecar_path, "w", encoding="utf-8") as f:
    json.dump(sidecar, f, ensure_ascii=False, indent=2)

print(f"✅ Short sidecar → {sidecar_path}")
print(f"   {len(ep)} episodes · {os.path.getsize(sidecar_path)/1024:.1f} KB")

✅ Short appendix injected → C:\Users\PC1\OneDrive\A4\CSV\ORI-dashboard-28.html
   size: 106.0 KB
   up-days: 23  |  hit≤5: 52%  hit≤10: 52%  hit≤20: 65%
   MA gate: above 100% (4)  below 58% (19)
   avg spread on hits: 6.44%  (15 of 23 hits)
✅ Short sidecar → C:\Users\PC1\OneDrive\A4\CSV\ORI-sim-notes-short.json
   23 episodes · 9.0 KB


In [15]:
# CELL 15 — Sanity: 200-MA value + distance
import pandas as pd
ma200 = df_rcl_dash["price"].tail(200).mean()
latest = df_rcl_dash["price"].iloc[-1]
print(f"200-MA = ฿{ma200:.2f}")
print(f"Distance = {(latest - ma200) / ma200 * 100:+.2f}%")

200-MA = ฿1.81
Distance = -1.63%


In [16]:
# CELL 16 — Sanity: MA series at −1, −20, −60
print("MA today  :", ma_series[-1])
print("MA 20d ago:", ma_series[-20])
print("MA 60d ago:", ma_series[-60])

MA today  : 1.8096
MA 20d ago: 1.8305
MA 60d ago: 1.921


In [17]:
# CELL 17 — Sanity: price row count
df_rcl_dash.shape

(722, 6)

In [18]:
# CELL 18 — CSV export: path
file_name = '3-yrs-sales-data.csv'
output_file = os.path.join(dat_path, file_name)

In [19]:
# CELL 19 — CSV export: write file
df_rcl_dash.sort_values(['date'],ascending=[True]).to_csv(output_file, index=False)

In [20]:
# CELL 20 — Sidecar sanity check (generic)
import json
s = json.load(open(os.path.join(csv_path, f"{NAME}-sim-notes.json"), encoding="utf-8"))
assert "annualized_pct" not in s["thresholds"], "old annualized threshold still there"
assert "avg_realized_pct_on_hits" in s["summary"], "realized summary missing"
stale = [k for k in ("annualized_pct", "annualized_capped")
         if any(k in ep for ep in s["episodes"])]
print("stale keys:", stale or "none")
print("summary:", {k: s["summary"][k] for k in
                   ("avg_realized_pct_on_hits","n_hit_target",
                    "hit_rate_pct","median_rows_to_target")})
print("episodes:", len(s["episodes"]))

stale keys: none
summary: {'avg_realized_pct_on_hits': 7.4, 'n_hit_target': 26, 'hit_rate_pct': {'h5': 24.561403508771928, 'h10': 36.84210526315789, 'h20': 45.614035087719294}, 'median_rows_to_target': 4}
episodes: 57
